# 🧪 Notebook Thử Nghiệm: Chuyển Đổi, Làm Sạch & Chuẩn Hóa Dữ Liệu (Bronze ➔ Silver)

Notebook này dùng để **khảo sát dữ liệu thực tế** và **thử nghiệm từng bước logic làm sạch & chuẩn hóa** trước khi đóng gói thành các module Python `.py` chính thức:

1. 📂 **Khám phá & Nạp dữ liệu Master Bronze**: Đọc 2 file `bronze_listings_master.json` và `bronze_details_valid.json`.
2. 🔗 **Thử nghiệm Ghép Bảng (Join Listings + Details)**: Kết hợp thông tin trang danh mục (Logo, tag ngoài) và trang chi tiết (JD sâu).
3. 💰 **Thử nghiệm Chuẩn hóa Mức Lương**: Tách `salary_min_vnd`, `salary_max_vnd`, xử lý tỷ giá USD, phát hiện lương thỏa thuận.
4. 🎓 **Thử nghiệm Chuẩn hóa Cấp Bậc (Seniority)**: Phân loại `Intern`, `Fresher`, `Junior`, `Middle`, `Senior`, `Lead`, `Manager`.
5. 📍 **Thử nghiệm Chuẩn hóa Địa Điểm**: Gom các chuỗi địa chỉ dài về `Hà Nội`, `Hồ Chí Minh`, `Đà Nẵng`, `Remote`, v.v.
6. 🔄 **Thử nghiệm Khử Trùng Chéo Sàn (Deduplication)**: Phát hiện tin đăng trùng giữa TopCV, TopDev, VietnamWorks.
7. 📊 **Preview Bảng Dữ Liệu Silver Hoàn Chỉnh**: Xem trước bảng phẳng Tabular sẵn sàng xuất sang Parquet.

In [2]:
import os
import re
import json
import pandas as pd
import numpy as np
from collections import Counter
from IPython.display import display, HTML

# Cấu hình hiển thị pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)
print('Các thư viện đã sẵn sàng!')

Các thư viện đã sẵn sàng!


## 1. Nạp Dữ Liệu Từ Tầng Master Bronze

In [4]:
# Xác định đường dẫn file Bronze
possible_paths = [
    os.path.abspath(os.path.join("..", "data", "bronze")),
    os.path.abspath(os.path.join("data", "bronze"))
]
bronze_dir = next((p for p in possible_paths if os.path.exists(p)), None)

if not bronze_dir:
    raise FileNotFoundError("Không tìm thấy thư mục data/bronze!")

listings_path = os.path.join(bronze_dir, "bronze_listings_master.json")
details_path = os.path.join(bronze_dir, "bronze_details_valid.json")

print(f"📂 Đang đọc Listings từ: {listings_path}")
df_listings = pd.read_json(listings_path)

print(f"📂 Đang đọc Details từ : {details_path}")
df_details = pd.read_json(details_path)

print(f"\n📊 Tổng số tin Listings: {len(df_listings):,} dòng, {df_listings.shape[1]} cột")
print(f"📊 Tổng số tin Details : {len(df_details):,} dòng, {df_details.shape[1]} cột")

# Xem nhanh phân bố theo sàn
display(pd.DataFrame({
    'Số Listings': df_listings['source'].value_counts(),
    'Số Details Valid': df_details['source'].value_counts()
}))

📂 Đang đọc Listings từ: C:\Users\Minh Doan\repo_github\IT-Job-Market-Intelligence\data\bronze\bronze_listings_master.json
📂 Đang đọc Details từ : C:\Users\Minh Doan\repo_github\IT-Job-Market-Intelligence\data\bronze\bronze_details_valid.json

📊 Tổng số tin Listings: 1,746 dòng, 20 cột
📊 Tổng số tin Details : 1,698 dòng, 35 cột
              Số Listings  Số Details Valid
source                                     
topcv                 862               814
vietnamworks          764               764
topdev                120               120


## 2. Thử Nghiệm Ghép Bảng (Join Listings + Details)
Ghép theo cặp khóa `(source, source_job_id)` để giữ lại toàn bộ các trường bổ trợ của cả 2 bảng:

In [6]:
# Chọn các cột giá trị từ Listings mà Details có thể chưa có hoặc bổ sung
listing_cols = [
    'source', 'source_job_id', 'company_logo_url', 'skills_raw', 
    'posted_at_raw', 'raw_file_name'
]
df_listings_sub = df_listings[listing_cols].rename(columns={
    'skills_raw': 'listing_skills_raw',
    'raw_file_name': 'listing_file_name'
})

# Thực hiện Merge (Left join từ Details sang Listings)
df_merged = pd.merge(
    df_details,
    df_listings_sub,
    on=['source', 'source_job_id'],
    how='left'
)

match_rate = df_merged['listing_file_name'].notna().mean() * 100
print(f"✅ Tỷ lệ ghép nối thành công: {match_rate:.2f}% ({len(df_merged):,} dòng)")
display(df_merged[['source', 'source_job_id', 'title_raw', 'company_name_raw', 'company_logo_url', 'listing_file_name', 'raw_file_name']].head(3))

✅ Tỷ lệ ghép nối thành công: 100.00% (1,698 dòng)
  source  source_job_id  \
0  topcv        2314643   
1  topcv        2314547   
2  topcv        2321616   

                                                                  title_raw  \
0  Full-Stack Web Engineer Intern/Fresher  (AI Coding: PHP / Ruby / Python)   
1                                                      Kỹ Sư Hệ Thống Nhúng   
2            UI/UX Designer - Mobile App- [HCM] - Thu Nhập Lên Đến 30 Triệu   

              company_name_raw  \
0  Công ty TNHH ZIGExN VeNtura   
1     Công ty TNHH Điện tử R&P   
2        CÔNG TY TNHH FOXSCORE   

                                                                  company_logo_url  \
0  https://cdn-new.topcv.vn/unsafe/https://static.topcv.vn/company_logos/67c94e...   
1  https://cdn-new.topcv.vn/unsafe/https://static.topcv.vn/company_logos/cong-t...   
2  https://cdn-new.topcv.vn/unsafe/https://static.topcv.vn/company_logos/ELdaFF...   

  listing_file_name raw_file_name  
0     

## 3. Khảo Sát & Thử Nghiệm Chuẩn Hóa Mức Lương (Salary Normalization)
Mục tiêu:
- Phát hiện trạng thái thương lượng (`is_negotiable`).
- Tách `salary_min_vnd` và `salary_max_vnd`.
- Tự động quy đổi USD sang VND (tỷ giá ước tính 25,400 VND/USD).
- Xử lý các dạng lương: triệu, VND số đầy đủ (`14000000`), tiền tố `Tới 30 triệu`, `Từ 15 triệu`.

In [8]:
EXCHANGE_RATE_USD_VND = 25400

def parse_salary_experimental(salary_str):
    if not salary_str or pd.isna(salary_str):
        return {'min_vnd': None, 'max_vnd': None, 'currency': 'UNKNOWN', 'is_negotiable': True}
    
    sal_lower = str(salary_str).lower().strip()
    
    # 1. Kiểm tra thương lượng / thỏa thuận
    if any(w in sal_lower for w in ['thỏa thuận', 'thoả thuận', 'thương lượng', 'negotiable', 'cạnh tranh']):
        return {'min_vnd': None, 'max_vnd': None, 'currency': 'NEGOTIABLE', 'is_negotiable': True}
    
    clean_str = sal_lower.replace(',', '')
    
    # Tìm dạng range: number (kèm tr/m tùy chọn) - number
    match_range = re.search(r'(-?\d+(?:\.\d+)?)\s*(?:tr|m|k)?\s*(?:-|đến|to)\s*(-?\d+(?:\.\d+)?)', clean_str)
    
    val_min = None
    val_max = None
    
    if match_range:
        val_min = float(match_range.group(1))
        val_max = float(match_range.group(2))
    else:
        # Tìm dạng tiền tố: 'Tới 30 triệu', 'Lên đến 50M'
        match_upto = re.search(r'(?:tới|lên đến|up to|tối đa|<)\s*(\d+(?:\.\d+)?)', clean_str)
        if match_upto:
            val_max = float(match_upto.group(1))
        else:
            # Tìm dạng tiền tố: 'Từ 15 triệu', 'Trên 20M'
            match_from = re.search(r'(?:từ|trên|from|>)\s*(\d+(?:\.\d+)?)', clean_str)
            if match_from:
                val_min = float(match_from.group(1))
            else:
                # Tìm số đơn lẻ (ví dụ: '15000000')
                match_single = re.search(r'(\d+(?:\.\d+)?)', clean_str)
                if match_single:
                    val_min = float(match_single.group(1))
                    val_max = val_min
    
    if val_min is None and val_max is None:
        return {'min_vnd': None, 'max_vnd': None, 'currency': 'UNKNOWN', 'is_negotiable': True}
    
    # Đảm bảo min <= max nếu có cả 2
    if val_min is not None and val_max is not None and val_min > val_max:
        val_min, val_max = val_max, val_min
        
    # 2. Nhận diện đơn vị tiền tệ & Quy đổi về VND
    check_val = val_max if val_max is not None else val_min
    
    has_trieu = any(u in sal_lower for u in ['triệu', 'tr']) or re.search(r'\d+\s*(?:m|tr)\b', sal_lower)
    has_usd = any(u in sal_lower for u in ['usd', 'đô']) or ('$' in sal_lower and not has_trieu)
    
    if has_usd:
        currency = 'USD'
        multiplier = EXCHANGE_RATE_USD_VND
    elif check_val >= 500_000:
        currency = 'VND'
        multiplier = 1  # Đã là số tiền VND đầy đủ
    elif has_trieu or check_val <= 300:
        currency = 'VND'
        multiplier = 1_000_000  # Đơn vị triệu đồng
    else:
        currency = 'VND'
        multiplier = 1
        
    min_vnd = int(val_min * multiplier) if val_min is not None else None
    max_vnd = int(val_max * multiplier) if val_max is not None else None
    
    return {
        'min_vnd': min_vnd,
        'max_vnd': max_vnd,
        'currency': currency,
        'is_negotiable': False
    }

# Thử nghiệm trên danh sách các mức lương thực tế đa dạng trong dataset
sample_salaries = [
    "Thương lượng", "Thoả thuận", "25 - 40 triệu", "Tới 30 triệu", "Từ 15 triệu",
    "15000000", "$ 1,202-2,600 /tháng", "$ 1,000-2,000 /tháng", "Up to $3,500",
    "14000000 - 18000000 MONTH", "$ 13tr-17tr /tháng", "20tr-40tr ₫/tháng", "30 - 20 triệu"
]

test_results = []
for s in sample_salaries:
    res = parse_salary_experimental(s)
    test_results.append({
        'Chuỗi Gốc (salary_raw)': s,
        'Lương Min (VND)': f"{res['min_vnd']:,}" if res['min_vnd'] else "-",
        'Lương Max (VND)': f"{res['max_vnd']:,}" if res['max_vnd'] else "-",
        'Loại Tiền': res['currency'],
        'Thương Lượng?': res['is_negotiable']
    })

display(pd.DataFrame(test_results))

       Chuỗi Gốc (salary_raw) Lương Min (VND) Lương Max (VND)   Loại Tiền  \
0                Thương lượng               -               -  NEGOTIABLE   
1                  Thoả thuận               -               -  NEGOTIABLE   
2               25 - 40 triệu      25,000,000      40,000,000         VND   
3                Tới 30 triệu               -      30,000,000         VND   
4                 Từ 15 triệu      15,000,000               -         VND   
5                    15000000      15,000,000      15,000,000         VND   
6        $ 1,202-2,600 /tháng      30,530,800      66,040,000         USD   
7        $ 1,000-2,000 /tháng      25,400,000      50,800,000         USD   
8                Up to $3,500      88,900,000      88,900,000         USD   
9   14000000 - 18000000 MONTH      14,000,000      18,000,000         VND   
10         $ 13tr-17tr /tháng      13,000,000      17,000,000         VND   
11          20tr-40tr ₫/tháng      20,000,000      40,000,000         VND   

## 4. Khảo Sát & Thử Nghiệm Chuẩn Hóa Cấp Bậc (Seniority Extraction)
Bóc tách cấp bậc chuyên môn từ `title_raw` và `experience_raw` thành các nhóm chuẩn:
- `Intern`, `Fresher`, `Junior`, `Middle`, `Senior`, `Lead`, `Manager/Director`

In [10]:
def extract_seniority_experimental(title_str, exp_str=""):
    text = f"{str(title_str)} {str(exp_str)}".lower()
    
    # Thứ tự ưu tiên kiểm tra
    if re.search(r'\b(director|giám đốc|vp|head of)\b', text):
        return 'Director'
    if re.search(r'\b(manager|trưởng phòng|quản lý)\b', text):
        return 'Manager'
    if re.search(r'\b(lead|leader|trưởng nhóm|chuyên gia cao cấp|principal|architect)\b', text):
        return 'Lead'
    if re.search(r'\b(senior|sr\.?|chuyên viên cao cấp|senior developer)\b', text):
        return 'Senior'
    if re.search(r'\b(junior|jr\.?|chuyên viên sơ cấp)\b', text):
        return 'Junior'
    if re.search(r'\b(fresher|mới tốt nghiệp)\b', text):
        return 'Fresher'
    if re.search(r'\b(intern|internship|thực tập|thực tập sinh)\b', text):
        return 'Intern'
    if re.search(r'\b(middle|mid|chuyên viên|kỹ sư)\b', text):
        return 'Middle'
    
    return 'Not Specified'  # Chưa phân cấp rõ

# Áp dụng thử nghiệm trên toàn bộ tin
df_merged['seniority_level'] = df_merged.apply(
    lambda row: extract_seniority_experimental(row['title_raw'], row['experience_raw']),
    axis=1
)

print("📊 Phân bố cấp bậc trong bộ dữ liệu thực tế:")
display(df_merged['seniority_level'].value_counts().to_frame('Số Lượng Tin'))

📊 Phân bố cấp bậc trong bộ dữ liệu thực tế:
                 Số Lượng Tin
seniority_level              
Not Specified             851
Middle                    278
Manager                   180
Senior                    174
Lead                      113
Intern                     37
Director                   25
Fresher                    21
Junior                     19


## 5. Khảo Sát & Thử Nghiệm Chuẩn Hóa Địa Điểm (Location Normalization)
Gom các chuỗi địa chỉ dài, chi tiết số nhà thành các khu vực trọng điểm:
- `Hà Nội`, `Hồ Chí Minh`, `Đà Nẵng`, `Cần Thơ`, `Hải Phòng`, `Remote`, `Khác`

In [12]:
def normalize_location_experimental(loc_str, work_mode_str=""):
    full_text = f"{str(loc_str)} {str(work_mode_str)}".lower()
    
    if 'remote' in full_text or 'từ xa' in full_text:
        return 'Remote'
    if any(k in full_text for k in ['hà nội', 'hanoi', 'ha noi', 'hn']):
        return 'Hà Nội'
    if any(k in full_text for k in ['hồ chí minh', 'ho chi minh', 'hcm', 'sài gòn', 'saigon', 'q.1', 'quận ']):
        return 'Hồ Chí Minh'
    if any(k in full_text for k in ['đà nẵng', 'da nang']):
        return 'Đà Nẵng'
    if any(k in full_text for k in ['hải phòng', 'hai phong']):
        return 'Hải Phòng'
    if any(k in full_text for k in ['cần thơ', 'can tho']):
        return 'Cần Thơ'
    if any(k in full_text for k in ['bình dương', 'binh duong']):
        return 'Bình Dương'
        
    return 'Khác / Chưa rõ'

df_merged['location_city'] = df_merged.apply(
    lambda row: normalize_location_experimental(row['location_raw'], row['work_mode_raw']),
    axis=1
)

print("📊 Phân bố địa điểm sau chuẩn hóa:")
display(df_merged['location_city'].value_counts().to_frame('Số Lượng Tin'))

📊 Phân bố địa điểm sau chuẩn hóa:
                Số Lượng Tin
location_city               
Hà Nội                  1040
Hồ Chí Minh              493
Khác / Chưa rõ           114
Đà Nẵng                   20
Hải Phòng                 17
Remote                    12
Bình Dương                 2


## 6. Thử Nghiệm Khử Trùng Lặp Chéo Sàn (Cross-Portal Deduplication)
Tìm các tin tuyển dụng được đăng tải đồng thời trên nhiều hơn 1 sàn tuyển dụng dựa trên **dấu vân tay** (Fingerprint): `Normalized Title + Normalized Company`:

In [14]:
def build_job_fingerprint(title, company):
    t_clean = re.sub(r'[^\w\s]', '', str(title).lower()).strip()
    c_clean = re.sub(r'[^\w\s]', '', str(company).lower()).strip()
    for prefix in ['công ty cổ phần', 'công ty tnhh', 'tập đoàn', 'company limited', 'jsc', 'ltd']:
        c_clean = c_clean.replace(prefix, '').strip()
    return f"{c_clean}___{t_clean}"

df_merged['dedup_fingerprint'] = df_merged.apply(
    lambda row: build_job_fingerprint(row['title_raw'], row['company_name_raw']),
    axis=1
)

# Tìm các fingerprint xuất hiện ở nhiều nguồn khác nhau
cross_dups = df_merged.groupby('dedup_fingerprint')['source'].nunique()
multi_source_fingerprints = cross_dups[cross_dups > 1].index

print(f"🔍 Số lượng tin trùng lặp xuất hiện trên từ 2 sàn trở lên: {len(multi_source_fingerprints):,} nhóm tin")

if len(multi_source_fingerprints) > 0:
    sample_fp = multi_source_fingerprints[0]
    sample_dups = df_merged[df_merged['dedup_fingerprint'] == sample_fp]
    print("\nVí dụ 1 tin trùng chéo sàn được phát hiện:")
    display(sample_dups[['source', 'title_raw', 'company_name_raw', 'salary_raw', 'location_city']])

🔍 Số lượng tin trùng lặp xuất hiện trên từ 2 sàn trở lên: 12 nhóm tin

Ví dụ 1 tin trùng chéo sàn được phát hiện:
           source                          title_raw  \
778        topdev  System Security Engineer (Senior)   
874  vietnamworks  System Security Engineer (Senior)   

                   company_name_raw    salary_raw location_city  
778  CyberLogitec Vietnam Co., Ltd.  Thương lượng        Remote  
874  CyberLogitec Vietnam Co., Ltd.  Thương lượng   Hồ Chí Minh  


## 7. Preview Bảng Dữ Liệu Silver Hoàn Chỉnh
Tổng hợp tất cả các trường đã chuẩn hóa vào DataFrame Silver hoàn chỉnh trong bộ nhớ (hoàn toàn không lưu file ra ổ đĩa) để bạn quan sát, phân tích và đánh giá:

In [16]:
# Áp dụng toàn bộ chuẩn hóa lương vào DataFrame
salary_parsed_list = df_merged['salary_raw'].apply(parse_salary_experimental)
df_silver = df_merged.copy()

df_silver['salary_min_vnd'] = [s['min_vnd'] for s in salary_parsed_list]
df_silver['salary_max_vnd'] = [s['max_vnd'] for s in salary_parsed_list]
df_silver['salary_currency'] = [s['currency'] for s in salary_parsed_list]
df_silver['is_negotiable'] = [s['is_negotiable'] for s in salary_parsed_list]

# Lựa chọn các cột đại diện cho bảng Silver sạch
silver_display_cols = [
    'source', 'title_raw', 'seniority_level', 'company_name_raw',
    'location_city', 'salary_raw', 'salary_min_vnd', 'salary_max_vnd',
    'is_negotiable', 'listing_file_name', 'raw_file_name'
]

print(f"✨ Đã xây dựng thành công bảng Silver mẫu ({len(df_silver):,} dòng trong bộ nhớ - KHÔNG XUẤT FILE)!")
display(df_silver[silver_display_cols].sample(10, random_state=42))

# Thống kê nhanh phân bố lương của các tin có số cụ thể
sal_stats = df_silver[~df_silver['is_negotiable']][['salary_min_vnd', 'salary_max_vnd']].describe()
print("\n📊 Thống kê mức lương (VND) của các tin có khoảng lương cụ thể:")
display(sal_stats.map(lambda x: f'{x:,.0f}' if pd.notna(x) else '-'))

✨ Đã xây dựng thành công bảng Silver mẫu (1,698 dòng trong bộ nhớ - KHÔNG XUẤT FILE)!
            source  \
618          topcv   
115          topcv   
135          topcv   
350          topcv   
1395  vietnamworks   
1447  vietnamworks   
669          topcv   
1686  vietnamworks   
1448  vietnamworks   
772         topdev   

                                                                            title_raw  \
618                                 Manual Software Testing (Saas,Product E-Commerce)   
115   Nhân Viên Kinh Doanh Thị Trường / Triển Khai Phần Mềm - Từ 1 Năm Kinh Nghiệm...   
135                               IT Sales - Tiếng Nhật - Open For Fresher [HCM - HN]   
350                                                 Lập Trình Viên .NET ( Full Level)   
1395                                                            Business Analyst (BA)   
1447  Giám Đốc Chi Nhánh Hồ Chí Minh (An Ninh Quốc Phòng/ Chính Phủ/ CNTT/ Thiết B...   
669                                            T

## 8. Mô Hình Hóa & Phân Tách 2 Bảng Cốt Lõi: `companies` và `jobs`

Theo mô hình **Silver Layer (Normalized Relational Model)**:
1. 🏢 **Bảng `companies` (Dimension)**: Phân tách thực thể công ty thành bảng danh mục riêng biệt. Khử trùng các biến thể tên viết hoa/thường, loại bỏ tiền tố pháp lý (*Công ty Cổ phần, TNHH...*), bóc tách quy mô nhân viên (*min/max*), ngành nghề, và cấp phát `company_id` duy nhất (`UUIDv5` cố định).
2. 💼 **Bảng `jobs` (Fact / Core)**: Bảng thực thể trung tâm chứa đầy đủ thông tin chuẩn hóa của tin tuyển dụng (chức danh, cấp bậc, lương `min/max/currency/type`, kinh nghiệm `min/max năm`, trình độ học vấn, hình thức làm việc `work_mode`), và liên kết khóa ngoại `company_id` sang bảng `companies`.

In [18]:
# --- 8.1. Xây Dựng Bảng Dimension: companies ---
import uuid

def clean_company_name(name):
    if not name or pd.isna(name):
        return "Unknown Company"
    name = str(name).strip()
    prefixes = [
        r"^CÔNG TY TNHH MTV\s+", r"^CÔNG TY TNHH\s+", r"^Công ty TNHH MTV\s+", r"^Công ty TNHH\s+",
        r"^CÔNG TY CỔ PHẦN\s+", r"^Công ty Cổ phần\s+", r"^Công Ty Cổ Phần\s+", r"^Công Ty TNHH\s+",
        r"^Tập đoàn\s+", r"^TẬP ĐOÀN\s+", r"^Chi nhánh\s+",
    ]
    for p in prefixes:
        name = re.sub(p, "", name, flags=re.IGNORECASE)
    suffixes = [
        r"\s+-\s+Chi nhánh.*$", r"\s+-\s+CN.*$", r"\s+JSC$", r"\s+Co\.,\s*Ltd\.?$",
    ]
    for s in suffixes:
        name = re.sub(s, "", name, flags=re.IGNORECASE)
    return name.strip()

def parse_company_size(size_str):
    if not size_str or pd.isna(size_str):
        return None, None
    s = str(size_str).strip()
    range_match = re.search(r"(\d+)\s*[-–]\s*(\d+)", s)
    if range_match:
        return int(range_match.group(1)), int(range_match.group(2))
    plus_match = re.search(r"(\d+)\s*\+", s)
    if plus_match:
        return int(plus_match.group(1)), None
    more_match = re.search(r"(?:hơn|trên)\s*(\d+)", s, re.IGNORECASE)
    if more_match:
        return int(more_match.group(1)), None
    less_match = re.search(r"(?:ít hơn|dưới|<\s*)(\d+)", s, re.IGNORECASE)
    if less_match:
        return 1, int(less_match.group(1))
    return None, None

# Trích xuất và khử trùng thực thể công ty
company_records = []
company_key_to_id = {}

for idx, row in df_merged.iterrows():
    raw_name = str(row.get('company_name_raw', '')).strip()
    norm_name = clean_company_name(raw_name)
    if not norm_name or norm_name == "Unknown Company":
        continue
    
    comp_key = norm_name.lower()
    if comp_key not in company_key_to_id:
        # Sinh UUIDv5 deterministic theo tên công ty chuẩn hóa
        comp_id = str(uuid.uuid5(uuid.NAMESPACE_DNS, f"company:{comp_key}"))[:18]
        company_key_to_id[comp_key] = comp_id
        
        comp_url = str(row.get('company_url', '') or '').strip()
        comp_logo = str(row.get('company_logo_url', '') or '').strip()
        comp_size_raw = str(row.get('company_size_raw', '') or '').strip()
        comp_loc_raw = str(row.get('company_location_raw', '') or '').strip()
        industry_raw = str(row.get('industry_raw', '') or '').strip()
        
        size_min, size_max = parse_company_size(comp_size_raw)
        
        company_records.append({
            'company_id': comp_id,
            'company_name_raw': raw_name,
            'company_name_normalized': norm_name,
            'company_url': comp_url,
            'company_logo_url': comp_logo,
            'company_size_raw': comp_size_raw,
            'company_size_min': size_min,
            'company_size_max': size_max,
            'company_location_raw': comp_loc_raw,
            'company_city': normalize_location_experimental(comp_loc_raw),
            'industry_raw': industry_raw,
            'industry_normalized': industry_raw if industry_raw else "Công nghệ thông tin / IT"
        })

df_companies = pd.DataFrame(company_records).drop_duplicates(subset=['company_id'])
print(f"🏢 Bảng companies: Đã gom nhóm và chuẩn hóa thành công {len(df_companies):,} công ty độc nhất từ {len(df_merged):,} tin!")
display(df_companies[['company_id', 'company_name_raw', 'company_name_normalized', 'company_size_min', 'company_size_max', 'company_city', 'industry_normalized']].head(5))


🏢 Bảng companies: Đã gom nhóm và chuẩn hóa thành công 890 công ty độc nhất từ 1,698 tin!
           company_id                              company_name_raw  \
0  b361832f-901f-5e4c                   Công ty TNHH ZIGExN VeNtura   
1  607e1460-1cd0-59f6                      Công ty TNHH Điện tử R&P   
2  b155e69a-30db-5d6b                         CÔNG TY TNHH FOXSCORE   
3  20f3cd62-0d47-582a  CÔNG TY CỔ PHẦN THƯƠNG MẠI QUỐC TẾ LB GLOBAL   
4  10d16130-e0cf-5de2                CÔNG TY TNHH VODAPLAY VIỆT NAM   

        company_name_normalized  company_size_min  company_size_max  \
0                ZIGExN VeNtura             100.0             499.0   
1                   Điện tử R&P              25.0              99.0   
2                      FOXSCORE              25.0              99.0   
3  THƯƠNG MẠI QUỐC TẾ LB GLOBAL              25.0              99.0   
4             VODAPLAY VIỆT NAM              25.0              99.0   

     company_city                                   indus

In [19]:
# --- 8.2. Xây Dựng Bảng Core: jobs ---

def clean_job_title(title):
    if not title or pd.isna(title):
        return ""
    t = str(title).strip()
    t = re.sub(r'\[.*?\]', '', t)  # Bỏ [HCM], [Hà Nội], v.v.
    # Bỏ các cụm từ quảng cáo thu nhập / đãi ngộ ở đuôi
    noise_patterns = [
        r'(?:-|\|)?\s*(?:thu nhập|lương|up to|upto|lên đến|tới)\s*.*$',
        r'(?:-|\|)?\s*(?:chế độ tốt|hấp dẫn|hot job|urgent|thưởng).*$',
    ]
    for np in noise_patterns:
        t = re.sub(np, '', t, flags=re.IGNORECASE)
    return re.sub(r'\s+', ' ', t).strip(' -|/,')

def parse_experience_years(exp_str):
    if not exp_str or pd.isna(exp_str):
        return None, None
    s = str(exp_str).strip().lower()
    if any(k in s for k in ['không yêu cầu', 'chưa có kinh nghiệm', 'không cần']):
        return 0.0, 0.0
    m_range = re.search(r"(\d+(?:\.\d+)?)\s*[-–]\s*(\d+(?:\.\d+)?)\s*năm", s)
    if m_range:
        return float(m_range.group(1)), float(m_range.group(2))
    m_min = re.search(r"(?:trên|từ|tối thiểu|\+)\s*(\d+(?:\.\d+)?)\s*năm|(\d+(?:\.\d+)?)\s*(?:\+|năm trở lên)", s)
    if m_min:
        val = m_min.group(1) or m_min.group(2)
        return float(val), None
    m_max = re.search(r"dưới\s*(\d+(?:\.\d+)?)\s*năm", s)
    if m_max:
        return 0.0, float(m_max.group(1))
    m_exact = re.search(r"(\d+(?:\.\d+)?)\s*năm", s)
    if m_exact:
        return float(m_exact.group(1)), float(m_exact.group(1))
    return None, None

def normalize_education(edu_str):
    if not edu_str or pd.isna(edu_str):
        return "Không yêu cầu"
    s = str(edu_str).strip().lower()
    if 'đại học' in s or 'cử nhân' in s or 'bachelor' in s:
        return "Đại học"
    if 'cao đẳng' in s or 'college' in s:
        return "Cao đẳng"
    if 'thạc sĩ' in s or 'master' in s:
        return "Thạc sĩ"
    if 'tiến sĩ' in s or 'phd' in s:
        return "Tiến sĩ"
    if 'trung cấp' in s or 'chứng chỉ' in s:
        return "Trung cấp / Chứng chỉ"
    return "Không yêu cầu"

def normalize_employment_type(emp_str):
    if not emp_str or pd.isna(emp_str):
        return "Full-time"
    s = str(emp_str).strip().lower()
    if any(k in s for k in ['toàn thời gian', 'full-time', 'full time']):
        return "Full-time"
    if any(k in s for k in ['bán thời gian', 'part-time', 'part time']):
        return "Part-time"
    if any(k in s for k in ['thực tập', 'internship', 'intern']):
        return "Internship"
    if any(k in s for k in ['hợp đồng', 'contract', 'freelance']):
        return "Contract"
    return "Full-time"

job_records = []

for idx, row in df_merged.iterrows():
    source = row.get('source', '')
    source_job_id = row.get('source_job_id', '')
    raw_name = str(row.get('company_name_raw', '')).strip()
    norm_name = clean_company_name(raw_name)
    comp_id = company_key_to_id.get(norm_name.lower(), None)
    
    # Deterministic UUIDv5 cho job_id
    job_uuid = str(uuid.uuid5(uuid.NAMESPACE_DNS, f"{source}:{source_job_id}"))[:18]
    
    # Chuẩn hóa mức lương & loại lương
    sal_parsed = parse_salary_experimental(row.get('salary_raw', ''))
    s_min = sal_parsed['min_vnd']
    s_max = sal_parsed['max_vnd']
    s_curr = "VND"
    s_period = "month"
    raw_sal = str(row.get('salary_raw', '')).lower()
    if sal_parsed['is_negotiable'] or any(k in raw_sal for k in ['thỏa thuận', 'thoả thuận', 'cạnh tranh', 'negotiable']):
        s_type = "negotiable"
    elif s_min is not None and s_max is not None:
        s_type = "fixed" if s_min == s_max else "range"
    elif s_max is not None:
        s_type = "max_only"
    elif s_min is not None:
        s_type = "min_only"
    else:
        s_type = "undisclosed"
    
    # Kinh nghiệm, học vấn, cấp bậc, hình thức
    e_min, e_max = parse_experience_years(row.get('experience_raw', ''))
    edu_level = normalize_education(row.get('education_raw', ''))
    seniority = extract_seniority_experimental(row.get('title_raw', ''), row.get('experience_raw', ''))
    city = normalize_location_experimental(row.get('location_raw', ''), row.get('work_mode_raw', ''))
    emp_type = normalize_employment_type(row.get('employment_type_raw', ''))
    
    wm_raw = (str(row.get('work_mode_raw', '')) + ' ' + str(row.get('location_raw', ''))).lower()
    work_mode = "Remote" if any(k in wm_raw for k in ['remote', 'từ xa']) else ("Hybrid" if 'hybrid' in wm_raw else "Onsite")
    
    job_records.append({
        'job_id': job_uuid,
        'source': source,
        'source_job_id': source_job_id,
        'job_url': row.get('job_url', ''),
        'title_raw': row.get('title_raw', ''),
        'title_normalized': clean_job_title(row.get('title_raw', '')),
        'seniority': seniority,
        'company_id': comp_id,
        'location_raw': row.get('location_raw', ''),
        'city': city,
        'salary_raw': row.get('salary_raw', ''),
        'salary_min': s_min,
        'salary_max': s_max,
        'salary_currency': s_curr,
        'salary_period': s_period,
        'salary_type': s_type,
        'experience_raw': row.get('experience_raw', ''),
        'experience_min_years': e_min,
        'experience_max_years': e_max,
        'education_raw': row.get('education_raw', ''),
        'education_level': edu_level,
        'work_mode_raw': row.get('work_mode_raw', ''),
        'work_mode': work_mode,
        'employment_type': emp_type,
        'working_time_raw': row.get('working_time_raw', ''),
        'published_at': row.get('published_at_raw', ''),
        'updated_at': row.get('updated_at_raw', ''),
        'deadline': row.get('deadline_raw', ''),
        'first_seen_at': row.get('crawled_at', ''),
        'last_seen_at': row.get('crawled_at', ''),
        'is_active': True,
        'data_source_id': source,
        'dq_status': row.get('dq_status', 'valid')
    })

df_jobs = pd.DataFrame(job_records)
print(f"💼 Bảng jobs: Đã tạo {len(df_jobs):,} bản ghi việc làm chuẩn hóa!")

# Kiểm tra tính toàn vẹn khóa ngoại (FK Integrity)
valid_fks = df_jobs['company_id'].isin(df_companies['company_id']).sum()
print(f"🔗 Kiểm tra toàn vẹn khóa ngoại (FK Integrity): {valid_fks}/{len(df_jobs)} ({valid_fks/len(df_jobs)*100:.1f}%) tin có company_id khớp chuẩn với bảng companies!")

print("\n🔍 So sánh Title Gốc vs Title Chuẩn Hóa:")
display(df_jobs[['title_raw', 'title_normalized', 'seniority']].sample(5, random_state=42))

print("\n🔍 Preview 5 dòng dữ liệu của bảng jobs:")
display(df_jobs[['job_id', 'title_normalized', 'seniority', 'company_id', 'city', 'salary_min', 'salary_max', 'salary_type', 'experience_min_years', 'education_level', 'work_mode']].head(5))

print("\n📊 Phân bố loại lương (salary_type):")
display(df_jobs['salary_type'].value_counts())

print("\n📊 Phân bố cấp bậc (seniority):")
display(df_jobs['seniority'].value_counts())

print("\n📊 Phân bố trình độ học vấn (education_level):")
display(df_jobs['education_level'].value_counts())


💼 Bảng jobs: Đã tạo 1,698 bản ghi việc làm chuẩn hóa!
🔗 Kiểm tra toàn vẹn khóa ngoại (FK Integrity): 1698/1698 (100.0%) tin có company_id khớp chuẩn với bảng companies!

🔍 So sánh Title Gốc vs Title Chuẩn Hóa:
                                                                            title_raw  \
618                                 Manual Software Testing (Saas,Product E-Commerce)   
115   Nhân Viên Kinh Doanh Thị Trường / Triển Khai Phần Mềm - Từ 1 Năm Kinh Nghiệm...   
135                               IT Sales - Tiếng Nhật - Open For Fresher [HCM - HN]   
350                                                 Lập Trình Viên .NET ( Full Level)   
1395                                                            Business Analyst (BA)   

                                                                  title_normalized  \
618                              Manual Software Testing (Saas,Product E-Commerce)   
115   Nhân Viên Kinh Doanh Thị Trường / Triển Khai Phần Mềm - Từ 1 Năm Kinh Nghiệm 

### 8.3. Xây Dựng Bảng `job_descriptions` (Tách Biệt Nội Dung Dài & Chuẩn Bị Cho NLP)

Theo đề xuất kiến trúc:
- **Mục đích**: Tách các cột văn bản dài (`overview`, `responsibilities`, `requirements`, `benefits`) ra khỏi bảng `jobs` để tối ưu dung lượng và tốc độ truy vấn nghiệp vụ.
- **Làm sạch (`clean`)**: Loại bỏ thẻ HTML, chuẩn hóa ký tự đầu dòng (bullet points `•`, `-`), chuẩn hóa khoảng trắng.
- **Phát hiện ngôn ngữ (`language`)**: Nhận diện tin viết bằng tiếng Việt (`vi`), tiếng Anh (`en`) hay song ngữ (`vi-en`) phục vụ chọn model NLP sau này.
- **Băm nội dung (`text_hash`)**: Dùng mã băm SHA256 của toàn bộ nội dung sạch để nhận diện tin có bị sửa đổi nội dung giữa các lần crawl hay không.

In [21]:
# --- 8.3. Xây Dựng Bảng job_descriptions ---
import hashlib
from datetime import datetime

def clean_html_text(text):
    if not text or pd.isna(text):
        return ""
    s = str(text)
    # Loại bỏ các thẻ HTML rác
    s = re.sub(r'<[^>]+>', ' ', s)
    # Thay thế HTML entities
    s = s.replace('&nbsp;', ' ').replace('&amp;', '&').replace('&lt;', '<').replace('&gt;', '>')
    # Chuẩn hóa các dấu gạch đầu dòng
    s = re.sub(r'[•·▪⁃►–—]', '\n- ', s)
    # Chuẩn hóa khoảng trắng & xuống dòng
    s = re.sub(r'[ \t]+', ' ', s)
    s = re.sub(r'\n\s*\n+', '\n', s)
    return s.strip()

def detect_language(text):
    if not text or len(text.strip()) < 20:
        return 'unknown'
    s = text.lower()
    vn_chars = 'àáảãạăằắẳẵặâầấẩẫậèéẻẽẹêềếểễệìíỉĩịòóỏõọôồốổỗộơờớởỡợùúủũụưừứửữựỳýỷỹỵđ'
    vn_count = sum(1 for c in s if c in vn_chars)
    total_alpha = sum(1 for c in s if c.isalpha())
    if total_alpha == 0:
        return 'unknown'
    vn_ratio = vn_count / total_alpha
    if vn_ratio > 0.04:
        en_words = ['experience', 'requirements', 'skills', 'responsibilities', 'developer', 'engineer', 'salary', 'benefits']
        en_found = sum(1 for w in en_words if w in s)
        if en_found >= 3:
            return 'vi-en'
        return 'vi'
    elif vn_count == 0:
        return 'en'
    else:
        return 'vi-en'

jd_records = []
now_iso = datetime.now().isoformat()

for idx, row in df_merged.iterrows():
    source = row.get('source', '')
    source_job_id = row.get('source_job_id', '')
    # Khóa ngoại và khóa chính khớp 1-1 với jobs table
    job_uuid = str(uuid.uuid5(uuid.NAMESPACE_DNS, f"{source}:{source_job_id}"))[:18]
    
    overview_raw = str(row.get('overview_raw', '') or '')
    responsibilities_raw = str(row.get('responsibilities_raw', '') or '')
    requirements_raw = str(row.get('requirements_raw', '') or '')
    benefits_raw = str(row.get('benefits_raw', '') or '')
    
    # Làm sạch văn bản
    desc_clean = clean_html_text(overview_raw + "\n" + responsibilities_raw)
    req_clean = clean_html_text(requirements_raw)
    ben_clean = clean_html_text(benefits_raw)
    
    # Nhận diện ngôn ngữ từ nội dung gộp
    full_text = desc_clean + " " + req_clean
    lang = detect_language(full_text)
    
    # Băm text_hash để kiểm soát versioning / thay đổi nội dung
    text_hash = hashlib.sha256((full_text + " " + ben_clean).encode('utf-8')).hexdigest()[:16]
    
    jd_records.append({
        'job_id': job_uuid,
        'overview_raw': overview_raw,
        'responsibilities_raw': responsibilities_raw,
        'requirements_raw': requirements_raw,
        'benefits_raw': benefits_raw,
        'description_clean': desc_clean,
        'requirements_clean': req_clean,
        'benefits_clean': ben_clean,
        'language': lang,
        'text_hash': text_hash,
        'cleaned_at': now_iso
    })

df_job_descriptions = pd.DataFrame(jd_records)
print(f"📄 Bảng job_descriptions: Đã xử lý {len(df_job_descriptions):,} bản ghi JD văn bản!")

# Kiểm tra toàn vẹn quan hệ 1-1 với bảng jobs
matched_with_jobs = df_job_descriptions['job_id'].isin(df_jobs['job_id']).sum()
print(f"🔗 Kiểm tra toàn vẹn 1-1 với bảng jobs: {matched_with_jobs}/{len(df_job_descriptions)} ({matched_with_jobs/len(df_job_descriptions)*100:.1f}%) khớp tuyệt đối!")

print("\n📊 Phân bố ngôn ngữ tin tuyển dụng (language):")
display(df_job_descriptions['language'].value_counts())

print("\n🔍 Preview 3 dòng bảng job_descriptions (các trường làm sạch & metadata):")
display(df_job_descriptions[['job_id', 'language', 'text_hash', 'cleaned_at', 'description_clean', 'requirements_clean']].head(3))

print("\n💾 So sánh dung lượng trong bộ nhớ RAM:")
size_jobs_kb = df_jobs.memory_usage(deep=True).sum() / 1024
size_jd_kb = df_job_descriptions.memory_usage(deep=True).sum() / 1024
print(f"  - Bảng jobs (nhẹ, phục vụ filter/thống kê): {size_jobs_kb:,.1f} KB")
print(f"  - Bảng job_descriptions (nặng, phục vụ NLP/Search): {size_jd_kb:,.1f} KB (Gấp {size_jd_kb/size_jobs_kb:.1f} lần bảng jobs)")


📄 Bảng job_descriptions: Đã xử lý 1,698 bản ghi JD văn bản!
🔗 Kiểm tra toàn vẹn 1-1 với bảng jobs: 1698/1698 (100.0%) khớp tuyệt đối!

📊 Phân bố ngôn ngữ tin tuyển dụng (language):
language
vi       1241
en        377
vi-en      80
Name: count, dtype: int64

🔍 Preview 3 dòng bảng job_descriptions (các trường làm sạch & metadata):
               job_id language         text_hash                  cleaned_at  \
0  632db9af-f938-516f       en  13d6a9b63a1b5423  2026-10-12T16:25:43.800077   
1  59053e61-8651-55d8       vi  49f1a785c776dd1b  2026-10-12T16:25:43.800077   
2  1d1cefc2-ca21-59db       vi  a7c536699cc8c980  2026-10-12T16:25:43.800077   

                                                                 description_clean  \
0  COMPANY INTRODUCTION\nAbout our parent company: ZIGExN Co.,Ltd\n- ZIGEXN is ...   
1  R&P là công ty công nghệ, hoạt động trên lĩnh vực điện tử và công nghệ thông...   
2  FOXSCORE \n- NỀN TẢNG THỂ THAO THÔNG MINH\nRa đời năm 2025, FOXSCORE là nền ...   

  

### 8.4. Xây Dựng Bảng Danh Mục Kỹ Năng Chuẩn: `skills`

Danh mục kỹ năng CNTT chuẩn dùng chung cho toàn hệ thống (`skills` Dimension):
- `skill_id`: Khóa chính định danh duy nhất.
- `skill_name`: Tên kỹ năng chuẩn (Python, SQL, Docker, AWS, React...).
- `skill_normalized`: Tên viết thường phục vụ tìm kiếm/khớp lệnh (python, sql...).
- `skill_category`: Phân loại nhóm kỹ năng (*Programming Language, Framework / Library, Database, Cloud & DevOps, Data Engineering, AI / Machine Learning, BI & Analytics, Testing & QA, Architecture & Methodology*).
- `description`: Mô tả chi tiết kỹ năng.
- `is_active`: Trạng thái hoạt động (`True`).

In [23]:
# --- 8.4. Xây Dựng Bảng skills (Master Skills Taxonomy) ---

SKILLS_TAXONOMY = [
    # Programming Languages
    ('Python', 'python', 'Programming Language', r'\bpython\b', 'Ngôn ngữ lập trình Python'),
    ('Java', 'java', 'Programming Language', r'\bjava\b(?!script)', 'Ngôn ngữ lập trình Java'),
    ('JavaScript', 'javascript', 'Programming Language', r'\b(?:javascript|js)\b', 'Ngôn ngữ lập trình JavaScript'),
    ('TypeScript', 'typescript', 'Programming Language', r'\b(?:typescript|ts)\b', 'Ngôn ngữ lập trình TypeScript'),
    ('C#', 'c#', 'Programming Language', r'\bc#\b|\bc\s*sharp\b', 'Ngôn ngữ lập trình C#'),
    ('C++', 'c++', 'Programming Language', r'\bc\+\+\b', 'Ngôn ngữ lập trình C++'),
    ('PHP', 'php', 'Programming Language', r'\bphp\b', 'Ngôn ngữ lập trình PHP'),
    ('Go', 'go', 'Programming Language', r'\b(?:golang|go)\b', 'Ngôn ngữ lập trình Golang'),
    ('Ruby', 'ruby', 'Programming Language', r'\bruby\b', 'Ngôn ngữ lập trình Ruby'),
    ('Kotlin', 'kotlin', 'Programming Language', r'\bkotlin\b', 'Ngôn ngữ lập trình Kotlin'),
    ('Swift', 'swift', 'Programming Language', r'\bswift\b', 'Ngôn ngữ lập trình Swift'),
    ('Rust', 'rust', 'Programming Language', r'\brust\b', 'Ngôn ngữ lập trình Rust'),
    ('HTML/CSS', 'html/css', 'Programming Language', r'\bhtml\b|\bcss\b', 'Ngôn ngữ đánh dấu HTML & CSS'),
    
    # Frameworks / Web / Mobile
    ('React', 'react', 'Framework / Library', r'\b(?:react|reactjs|react\.js)\b', 'Thư viện React UI'),
    ('Vue.js', 'vue.js', 'Framework / Library', r'\b(?:vue|vuejs|vue\.js)\b', 'Framework Vue.js frontend'),
    ('Angular', 'angular', 'Framework / Library', r'\bangular\b', 'Framework Angular frontend'),
    ('Node.js', 'node.js', 'Framework / Library', r'\b(?:nodejs|node\.js)\b', 'Môi trường thực thi Node.js'),
    ('Next.js', 'next.js', 'Framework / Library', r'\b(?:nextjs|next\.js)\b', 'Framework Next.js fullstack'),
    ('Spring Boot', 'spring boot', 'Framework / Library', r'\bspring\s*boot\b|\bspring\s*framework\b', 'Framework Spring Boot Java'),
    ('.NET', '.net', 'Framework / Library', r'\b(?:\.net|dotnet|asp\.net)\b', 'Framework .NET / ASP.NET'),
    ('Django', 'django', 'Framework / Library', r'\bdjango\b', 'Web framework Django Python'),
    ('FastAPI', 'fastapi', 'Framework / Library', r'\bfastapi\b', 'Web framework FastAPI Python'),
    ('Laravel', 'laravel', 'Framework / Library', r'\blaravel\b', 'Web framework Laravel PHP'),
    ('Flutter', 'flutter', 'Mobile Development', r'\bflutter\b', 'Framework đa nền tảng Flutter'),
    ('React Native', 'react native', 'Mobile Development', r'\breact\s*native\b', 'Framework đa nền tảng React Native'),
    
    # Databases
    ('SQL', 'sql', 'Database', r'\bsql\b', 'Ngôn ngữ truy vấn cơ sở dữ liệu SQL'),
    ('MySQL', 'mysql', 'Database', r'\bmysql\b', 'Hệ quản trị CSDL MySQL'),
    ('PostgreSQL', 'postgresql', 'Database', r'\b(?:postgresql|postgres)\b', 'Hệ quản trị CSDL PostgreSQL'),
    ('MongoDB', 'mongodb', 'Database', r'\bmongodb\b', 'CSDL NoSQL MongoDB'),
    ('Redis', 'redis', 'Database', r'\bredis\b', 'In-memory cache Redis'),
    ('Oracle', 'oracle', 'Database', r'\boracle\b', 'Hệ quản trị CSDL Oracle'),
    ('SQL Server', 'sql server', 'Database', r'\bsql\s*server\b|\bmssql\b', 'Hệ quản trị CSDL Microsoft SQL Server'),
    ('Elasticsearch', 'elasticsearch', 'Database', r'\belasticsearch\b', 'Công cụ tìm kiếm Elasticsearch'),
    
    # Cloud & DevOps
    ('AWS', 'aws', 'Cloud & DevOps', r'\b(?:aws|amazon\s*web\s*services)\b', 'Nền tảng đám mây Amazon AWS'),
    ('Azure', 'azure', 'Cloud & DevOps', r'\bazure\b', 'Nền tảng đám mây Microsoft Azure'),
    ('Google Cloud (GCP)', 'gcp', 'Cloud & DevOps', r'\b(?:gcp|google\s*cloud)\b', 'Nền tảng đám mây Google Cloud'),
    ('Docker', 'docker', 'Cloud & DevOps', r'\bdocker\b', 'Nền tảng đóng gói container Docker'),
    ('Kubernetes', 'kubernetes', 'Cloud & DevOps', r'\b(?:kubernetes|k8s)\b', 'Công cụ điều phối Kubernetes'),
    ('CI/CD', 'ci/cd', 'Cloud & DevOps', r'\bci\s*/\s*cd\b|\bci-cd\b', 'Quy trình tích hợp & triển khai liên tục'),
    ('Git', 'git', 'Cloud & DevOps', r'\bgit\b|\bgithub\b|\bgitlab\b', 'Hệ thống quản lý phiên bản Git'),
    ('Linux', 'linux', 'Cloud & DevOps', r'\blinux\b', 'Hệ điều hành Linux'),
    
    # Big Data & Data Eng
    ('Spark', 'spark', 'Data Engineering', r'\b(?:spark|pyspark)\b', 'Nền tảng xử lý phân tán Apache Spark'),
    ('Kafka', 'kafka', 'Data Engineering', r'\bkafka\b', 'Nền tảng streaming Apache Kafka'),
    ('Airflow', 'airflow', 'Data Engineering', r'\bairflow\b', 'Công cụ orchestration Apache Airflow'),
    ('Hadoop', 'hadoop', 'Data Engineering', r'\bhadoop\b', 'Hệ sinh thái Apache Hadoop'),
    ('Snowflake', 'snowflake', 'Data Engineering', r'\bsnowflake\b', 'Data warehouse đám mây Snowflake'),
    
    # AI / ML & Data Science
    ('Machine Learning', 'machine learning', 'AI / Machine Learning', r'\b(?:machine\s*learning|ml)\b', 'Học máy Machine Learning'),
    ('Deep Learning', 'deep learning', 'AI / Machine Learning', r'\bdeep\s*learning\b', 'Học sâu Deep Learning'),
    ('PyTorch', 'pytorch', 'AI / Machine Learning', r'\bpytorch\b', 'Framework học sâu PyTorch'),
    ('TensorFlow', 'tensorflow', 'AI / Machine Learning', r'\btensorflow\b', 'Framework học sâu TensorFlow'),
    ('Computer Vision', 'computer vision', 'AI / Machine Learning', r'\bcomputer\s*vision\b|\bopencv\b', 'Thị giác máy tính'),
    ('NLP', 'nlp', 'AI / Machine Learning', r'\b(?:nlp|natural\s*language\s*processing)\b', 'Xử lý ngôn ngữ tự nhiên NLP'),
    ('LLM', 'llm', 'AI / Machine Learning', r'\b(?:llm|large\s*language\s*model|genai|generative\s*ai)\b', 'Mô hình ngôn ngữ lớn LLM / GenAI'),
    
    # BI & Analytics
    ('Power BI', 'power bi', 'BI & Analytics', r'\bpower\s*bi\b', 'Công cụ trực quan hóa dữ liệu Power BI'),
    ('Tableau', 'tableau', 'BI & Analytics', r'\btableau\b', 'Công cụ trực quan hóa dữ liệu Tableau'),
    ('Excel', 'excel', 'BI & Analytics', r'\bexcel\b', 'Phân tích bảng tính Microsoft Excel'),
    
    # QA & Testing
    ('Automation Testing', 'automation testing', 'Testing & QA', r'\bautomation\s*test(?:ing)?\b', 'Kiểm thử tự động'),
    ('Manual Testing', 'manual testing', 'Testing & QA', r'\bmanual\s*test(?:ing)?\b', 'Kiểm thử thủ công'),
    ('Selenium', 'selenium', 'Testing & QA', r'\bselenium\b', 'Framework kiểm thử web Selenium'),
    ('Postman', 'postman', 'Testing & QA', r'\bpostman\b', 'Công cụ kiểm thử API Postman'),
    
    # Architecture & Methodology
    ('Microservices', 'microservices', 'Architecture & Methodology', r'\bmicroservices?\b', 'Kiến trúc vi dịch vụ Microservices'),
    ('RESTful API', 'restful api', 'Architecture & Methodology', r'\brest(?:ful)?\s*api\b|\brest\s*api\b', 'Kiến trúc API RESTful'),
    ('Agile/Scrum', 'agile/scrum', 'Architecture & Methodology', r'\bagile\b|\bscrum\b', 'Mô hình phát triển Agile/Scrum'),
    ('OOP', 'oop', 'Architecture & Methodology', r'\boop\b|\bobject\s*oriented\b', 'Lập trình hướng đối tượng OOP'),
]

skills_records = []
for idx, (name, norm, cat, pat, desc) in enumerate(SKILLS_TAXONOMY, start=101):
    skills_records.append({
        'skill_id': idx,
        'skill_name': name,
        'skill_normalized': norm,
        'skill_category': cat,
        'description': desc,
        'pattern': pat,
        'is_active': True
    })

df_skills = pd.DataFrame(skills_records)
print(f"📚 Bảng skills: Đã tạo danh mục chuẩn gồm {len(df_skills)} kỹ năng CNTT!")
display(df_skills[['skill_id', 'skill_name', 'skill_category', 'description', 'is_active']].head(8))


📚 Bảng skills: Đã tạo danh mục chuẩn gồm 64 kỹ năng CNTT!
   skill_id  skill_name        skill_category                    description  \
0       101      Python  Programming Language      Ngôn ngữ lập trình Python   
1       102        Java  Programming Language        Ngôn ngữ lập trình Java   
2       103  JavaScript  Programming Language  Ngôn ngữ lập trình JavaScript   
3       104  TypeScript  Programming Language  Ngôn ngữ lập trình TypeScript   
4       105          C#  Programming Language          Ngôn ngữ lập trình C#   
5       106         C++  Programming Language         Ngôn ngữ lập trình C++   
6       107         PHP  Programming Language         Ngôn ngữ lập trình PHP   
7       108          Go  Programming Language      Ngôn ngữ lập trình Golang   

   is_active  
0       True  
1       True  
2       True  
3       True  
4       True  
5       True  
6       True  
7       True  


### 8.5. Xây Dựng Bảng Liên Kết N - N: `job_skills`

Bảng liên kết nhiều - nhiều (Bridge / Associative Table) giữa **`jobs`** và **`skills`**:
- `job_id`: Khóa ngoại tới `jobs`.
- `skill_id`: Khóa ngoại tới `skills`.
- `skill_type`: `required` (bắt buộc), `preferred` (ưu tiên), `mentioned` (nhắc tới).
- `extraction_method`: `tag_direct` (tag từ sàn), `tag_listing` (tag từ listing), `rule_based_text` (trích xuất từ JD).
- `confidence`: Độ tin cậy của kết quả (0.70 - 1.0).
- `evidence_text`: Đoạn văn bản chứa từ khóa làm bằng chứng ngữ cảnh (Explainability).
- `extracted_at`: Thời điểm trích xuất.

In [25]:
# --- 8.5. Xây Dựng Bảng job_skills (Bridge Table) ---

job_skills_records = []
now_iso = datetime.now().isoformat()

# Mapping tags từ listings để tăng tốc độ
listing_tags_map = {}
for _, l_row in df_listings.iterrows():
    k = (l_row['source'], l_row['source_job_id'])
    listing_tags_map[k] = str(l_row.get('skills_raw') or '').lower()

for idx, row in df_merged.iterrows():
    source = row.get('source', '')
    source_job_id = row.get('source_job_id', '')
    job_uuid = str(uuid.uuid5(uuid.NAMESPACE_DNS, f"{source}:{source_job_id}"))[:18]
    
    req_tag = str(row.get('required_skills_raw', '') or '').lower()
    pref_tag = str(row.get('preferred_skills_raw', '') or '').lower()
    list_tag = listing_tags_map.get((source, source_job_id), '')
    
    req_text = str(row.get('requirements_raw', '') or '')
    desc_text = str(row.get('responsibilities_raw', '') or '') + ' ' + str(row.get('overview_raw', '') or '')
    
    found_skills_in_job = set()
    
    for _, s_row in df_skills.iterrows():
        s_id = s_row['skill_id']
        s_pat = s_row['pattern']
        s_name = s_row['skill_name']
        
        # Priority 1: Tag bắt buộc từ sàn (TopDev)
        if re.search(s_pat, req_tag, re.IGNORECASE):
            job_skills_records.append({
                'job_id': job_uuid,
                'skill_id': s_id,
                'skill_type': 'required',
                'extraction_method': 'tag_direct',
                'confidence': 1.0,
                'evidence_text': f'Tag yêu cầu: {s_name}',
                'extracted_at': now_iso
            })
            found_skills_in_job.add(s_id)
            continue
            
        # Priority 2: Tag ưu tiên từ sàn
        if re.search(s_pat, pref_tag, re.IGNORECASE):
            job_skills_records.append({
                'job_id': job_uuid,
                'skill_id': s_id,
                'skill_type': 'preferred',
                'extraction_method': 'tag_direct',
                'confidence': 1.0,
                'evidence_text': f'Tag ưu tiên: {s_name}',
                'extracted_at': now_iso
            })
            found_skills_in_job.add(s_id)
            continue
            
        # Priority 3: Tag từ listing
        if re.search(s_pat, list_tag, re.IGNORECASE):
            job_skills_records.append({
                'job_id': job_uuid,
                'skill_id': s_id,
                'skill_type': 'required',
                'extraction_method': 'tag_listing',
                'confidence': 0.95,
                'evidence_text': f'Tag listing: {s_name}',
                'extracted_at': now_iso
            })
            found_skills_in_job.add(s_id)
            continue
            
        # Priority 4: Quét trong requirements text
        m_req = re.search(s_pat, req_text, re.IGNORECASE)
        if m_req:
            start = max(0, m_req.start() - 25)
            end = min(len(req_text), m_req.end() + 25)
            snippet = req_text[start:end].replace('\n', ' ').strip()
            is_pref = any(k in snippet.lower() for k in ['ưu tiên', 'plus', 'preferred', 'nice to have'])
            stype = 'preferred' if is_pref else 'required'
            job_skills_records.append({
                'job_id': job_uuid,
                'skill_id': s_id,
                'skill_type': stype,
                'extraction_method': 'rule_based_text',
                'confidence': 0.85,
                'evidence_text': f'...{snippet}...',
                'extracted_at': now_iso
            })
            found_skills_in_job.add(s_id)
            continue
            
        # Priority 5: Quét trong description / overview text
        m_desc = re.search(s_pat, desc_text, re.IGNORECASE)
        if m_desc:
            start = max(0, m_desc.start() - 25)
            end = min(len(desc_text), m_desc.end() + 25)
            snippet = desc_text[start:end].replace('\n', ' ').strip()
            job_skills_records.append({
                'job_id': job_uuid,
                'skill_id': s_id,
                'skill_type': 'mentioned',
                'extraction_method': 'rule_based_text',
                'confidence': 0.70,
                'evidence_text': f'...{snippet}...',
                'extracted_at': now_iso
            })
            found_skills_in_job.add(s_id)

df_job_skills = pd.DataFrame(job_skills_records)
df_job_skills = df_job_skills.drop_duplicates(subset=['job_id', 'skill_id', 'skill_type'])

print(f"🔗 Bảng job_skills: Đã trích xuất thành công {len(df_job_skills):,} mối liên kết job ➔ skill!")

# Kiểm tra toàn vẹn khóa ngoại (FK Integrity)
fk_job_ok = df_job_skills['job_id'].isin(df_jobs['job_id']).all()
fk_skill_ok = df_job_skills['skill_id'].isin(df_skills['skill_id']).all()
print(f"✅ Kiểm tra toàn vẹn khóa ngoại (FK Check): Job FK OK? {fk_job_ok} | Skill FK OK? {fk_skill_ok}")

print("\n🔍 Preview 10 dòng đầu tiên của bảng job_skills:")
display(df_job_skills.head(10))

print("\n📊 Phân bố loại kỹ năng (skill_type):")
display(df_job_skills['skill_type'].value_counts())

print("\n📊 Phân bố phương thức trích xuất (extraction_method):")
display(df_job_skills['extraction_method'].value_counts())

print("\n🔥 TOP 15 KỸ NĂNG CÓ NHU CẦU CAO NHẤT THỊ TRƯỜNG IT:")
top_skills_df = df_job_skills.merge(df_skills, on='skill_id')['skill_name'].value_counts().head(15)
display(top_skills_df)


🔗 Bảng job_skills: Đã trích xuất thành công 3,651 mối liên kết job ➔ skill!
✅ Kiểm tra toàn vẹn khóa ngoại (FK Check): Job FK OK? True | Skill FK OK? True

🔍 Preview 10 dòng đầu tiên của bảng job_skills:
               job_id  skill_id skill_type extraction_method  confidence  \
0  632db9af-f938-516f       101   required   rule_based_text        0.85   
1  632db9af-f938-516f       102   required   rule_based_text        0.85   
2  632db9af-f938-516f       103   required   rule_based_text        0.85   
3  632db9af-f938-516f       107   required   rule_based_text        0.85   
4  632db9af-f938-516f       108   required   rule_based_text        0.85   
5  632db9af-f938-516f       109   required   rule_based_text        0.85   
6  632db9af-f938-516f       113   required   rule_based_text        0.85   
7  632db9af-f938-516f       127  mentioned   rule_based_text        0.70   
8  632db9af-f938-516f       128  mentioned   rule_based_text        0.70   
9  632db9af-f938-516f       129  men